# SmartCart - Notebook 3: Association Rule Mining (Apriori)

This notebook discovers frequent product combinations and association rules from user-level transactions.

## Objectives
- Build transaction baskets from cleaned interaction data
- Mine frequent itemsets using Apriori
- Generate association rules with support, confidence, and lift
- Present top patterns for bundle and cross-sell strategy
## Inputs
- `data/processed/user_data_clean.csv`

This notebook expects Notebook 1 to be executed first so processed files are available.

In [1]:
# Imports
from pathlib import Path
import sys
import subprocess
from itertools import combinations

import pandas as pd

In [2]:
# Load processed interaction data
cwd = Path.cwd()
processed_candidates = [cwd / 'data' / 'processed', cwd.parent / 'data' / 'processed']
processed_dir = next((p for p in processed_candidates if p.exists()), processed_candidates[0])
user_data_path = processed_dir / 'user_data_clean.csv'

if not user_data_path.exists():
    raise FileNotFoundError(
        'Missing data/processed/user_data_clean.csv. Run Notebook 1 first. '
        f'Current working directory: {cwd}'
    )

user_data = pd.read_csv(user_data_path)

print('Loaded:', user_data_path)
print('user_data shape:', user_data.shape)
display(user_data.head())

Loaded: /Users/solaris003/Repository/Projects/Recommender System & Pattern Mining for E-Commerce Analytics/data/processed/user_data_clean.csv
user_data shape: (724, 5)


,UserID,ProductID,Rating,Timestamp,Category
0,U000,P0020,1,2024-09-02,Home
1,U000,P0071,3,2024-09-02,Toys
2,U000,P0007,3,2024-09-03,Books
3,U000,P0044,2,2024-09-06,Books
4,U000,P0047,3,2024-09-06,Clothing


In [3]:
# Build transaction baskets (unique products per user)
transactions = (
    user_data.groupby('UserID')['ProductID']
    .apply(lambda s: sorted(set(s.astype(str))))
    .tolist()
)

num_transactions = len(transactions)
avg_basket_size = sum(len(t) for t in transactions) / num_transactions if num_transactions else 0

print(f'Number of transactions (users): {num_transactions}')
print(f'Average basket size: {avg_basket_size:.2f}')
print('Sample transaction:', transactions[0][:10] if num_transactions else [])

Number of transactions (users): 50
Average basket size: 14.48
Sample transaction: ['P0003', 'P0005', 'P0007', 'P0009', 'P0012', 'P0013', 'P0014', 'P0020', 'P0021', 'P0028']


In [4]:
# Use mlxtend when available; otherwise use a pure-Python fallback
try:
    from mlxtend.preprocessing import TransactionEncoder
    from mlxtend.frequent_patterns import apriori, association_rules
    USE_MLX = True
except Exception as e:
    USE_MLX = False
    print('mlxtend unavailable, using fallback Apriori implementation.')
    print('Reason:', type(e).__name__, str(e))

    class TransactionEncoder:
        def fit(self, transactions):
            items = sorted({item for t in transactions for item in t})
            self.columns_ = items
            return self

        def transform(self, transactions):
            item_index = {item: i for i, item in enumerate(self.columns_)}
            matrix = []
            for t in transactions:
                row = [False] * len(self.columns_)
                for item in t:
                    if item in item_index:
                        row[item_index[item]] = True
                matrix.append(row)
            return matrix

    def apriori(df, min_support=0.05, use_colnames=True, max_len=None):
        transactions_local = [set(df.columns[df.iloc[i].astype(bool)]) for i in range(len(df))]
        n_tx = len(transactions_local)

        if n_tx == 0:
            return pd.DataFrame(columns=['support', 'itemsets'])

        all_items = sorted({item for tx in transactions_local for item in tx})
        max_k = max_len if max_len is not None else len(all_items)

        rows = []
        for k in range(1, max_k + 1):
            for comb in combinations(all_items, k):
                itemset = set(comb)
                count = sum(1 for tx in transactions_local if itemset.issubset(tx))
                support = count / n_tx
                if support >= min_support:
                    rows.append({'support': support, 'itemsets': frozenset(itemset)})

        out = pd.DataFrame(rows)
        if out.empty:
            return pd.DataFrame(columns=['support', 'itemsets'])
        return out.sort_values('support', ascending=False).reset_index(drop=True)

    def association_rules(frequent_itemsets, metric='confidence', min_threshold=0.1):
        if frequent_itemsets.empty:
            return pd.DataFrame(columns=['antecedents', 'consequents', 'support', 'confidence', 'lift'])

        support_map = {frozenset(row['itemsets']): float(row['support']) for _, row in frequent_itemsets.iterrows()}
        rules_rows = []

        for itemset, sup_itemset in support_map.items():
            if len(itemset) < 2:
                continue

            items_list = sorted(itemset)
            for r in range(1, len(items_list)):
                for ant_tuple in combinations(items_list, r):
                    antecedent = frozenset(ant_tuple)
                    consequent = itemset - antecedent

                    sup_ant = support_map.get(antecedent)
                    sup_con = support_map.get(consequent)
                    if sup_ant is None or sup_con is None or sup_ant == 0 or sup_con == 0:
                        continue

                    confidence = sup_itemset / sup_ant
                    lift = confidence / sup_con

                    rules_rows.append({
                        'antecedents': antecedent,
                        'consequents': consequent,
                        'support': sup_itemset,
                        'confidence': confidence,
                        'lift': lift,
                    })

        rules_df = pd.DataFrame(rules_rows)
        if rules_df.empty:
            return pd.DataFrame(columns=['antecedents', 'consequents', 'support', 'confidence', 'lift'])

        if metric not in rules_df.columns:
            metric = 'confidence'
        return rules_df[rules_df[metric] >= min_threshold].reset_index(drop=True)

In [5]:
# One-hot encode transactions for Apriori
te = TransactionEncoder()
transaction_matrix = te.fit(transactions).transform(transactions)
transactions_df = pd.DataFrame(transaction_matrix, columns=te.columns_)

print('Encoded transaction matrix shape:', transactions_df.shape)
transactions_df.head()

Encoded transaction matrix shape: (50, 100)


,P0000,P0001,P0002,P0003,P0004,P0005,P0006,P0007,P0008,P0009,...,P0090,P0091,P0092,P0093,P0094,P0095,P0096,P0097,P0098,P0099
0,False,False,False,True,False,True,False,True,False,True,...,False,False,False,False,False,False,False,False,False,False
1,False,False,True,False,False,False,False,False,False,False,...,False,True,False,False,False,True,False,False,False,False
2,False,False,False,False,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
4,False,True,False,False,False,False,True,False,False,False,...,False,False,True,False,False,False,False,False,True,True


In [6]:
# Mine frequent itemsets and association rules
MIN_SUPPORT = 0.08
MAX_ITEMSET_LEN = 3
MIN_CONFIDENCE = 0.35

frequent_itemsets = apriori(
    transactions_df,
    min_support=MIN_SUPPORT,
    use_colnames=True,
    max_len=MAX_ITEMSET_LEN,
).sort_values('support', ascending=False).reset_index(drop=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)

rules = association_rules(
    frequent_itemsets,
    metric='confidence',
    min_threshold=MIN_CONFIDENCE,
)

if not rules.empty:
    rules = rules.sort_values(['lift', 'confidence', 'support'], ascending=False).reset_index(drop=True)
    rules = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
else:
    rules = pd.DataFrame(columns=['antecedents', 'consequents', 'support', 'confidence', 'lift'])

print(f'Frequent itemsets found: {len(frequent_itemsets)}')
print(f'Association rules found: {len(rules)}')

Frequent itemsets found: 192
Association rules found: 154


In [7]:
# Show top patterns
display(frequent_itemsets.head(15))
display(rules.head(15))

,support,itemsets,length
0,0.34,(P0070),1
1,0.26,(P0051),1
2,0.26,(P0089),1
3,0.24,(P0043),1
4,0.24,(P0088),1
5,0.22,(P0065),1
6,0.22,(P0004),1
7,0.22,(P0044),1
8,0.22,(P0077),1
9,0.22,(P0030),1


,antecedents,consequents,support,confidence,lift
0,"(P0022, P0070)",(P0015),0.08,1.000000,6.250000
1,"(P0039, P0070)",(P0015),0.08,1.000000,6.250000
2,(P0015),"(P0022, P0070)",0.08,0.500000,6.250000
3,(P0015),"(P0039, P0070)",0.08,0.500000,6.250000
4,"(P0015, P0070)",(P0022),0.08,0.800000,5.714286
5,"(P0079, P0044)",(P0013),0.08,0.800000,5.714286
6,"(P0015, P0070)",(P0039),0.08,0.800000,5.714286
7,(P0022),"(P0015, P0070)",0.08,0.571429,5.714286
8,(P0013),"(P0079, P0044)",0.08,0.571429,5.714286
9,(P0039),"(P0015, P0070)",0.08,0.571429,5.714286


## Interpretation Guide
- **Support**: how frequently an itemset appears across all transactions.
- **Confidence**: probability of the consequent given the antecedent.
- **Lift**: strength above random chance; lift > 1 indicates meaningful association.

High-lift, high-confidence rules are the strongest candidates for bundle design and cart cross-sell recommendations.